# 10 Answer Generation

Το notebook εκτελεί το στάδιο παραγωγής απαντήσεων πάνω στα αποτελέσματα retrieval για τα pipelines `dense`, `hybrid` και `hybrid_reranked`.

## Στόχος
- Φόρτωση ενός retrieval run.
- Δημιουργία context από τα ανακτημένα chunks.
- Παραγωγή απαντήσεων με OpenAI ή έλεγχος της ροής χωρίς κλήση API.
- Αποθήκευση QA results, manifest και συνοπτικών στατιστικών.

## Prompt
Το notebook περιέχει την τελική ρύθμιση του QA prompt. Οι αριθμητικές ερωτήσεις προτιμούν συμπαγείς απαντήσεις, ενώ οι ποιοτικές ερωτήσεις επιστρέφουν πλήρη πρόταση με υποστηρικτική λεπτομέρεια.


In [ ]:
import json
import time
import sys
import io
import os
from pathlib import Path

import pandas as pd
from tqdm import tqdm

# Διόρθωση κωδικοποίησης σε Windows
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "buffer"):
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")
    sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding="utf-8", errors="replace")


In [ ]:

RETRIEVAL_SOURCE = "dense"
# επιλογές:
# "dense"
# "hybrid"
# "hybrid_reranked"

# Το notebook παράγει αυτόματα QA outputs και για τα τρία retrieval sources,
# ώστε τα downstream notebooks να τρέχουν σειριακά από καθαρό data folder.
RETRIEVAL_SOURCES_TO_GENERATE = ["dense", "hybrid", "hybrid_reranked"]

TOP_K_CONTEXT = 5

GENERATION_MODE = "openai"
# Canonical runs must never overwrite real QA outputs with dry-run answers.
# Enable this only for an explicitly labelled smoke test.
FALLBACK_TO_DRY_RUN_WITHOUT_OPENAI_KEY = False
GENERATION_MODEL = "gpt-4o-mini"

USE_QUERY_LIMIT = False
QUERY_LIMIT = 20

SLEEP_BETWEEN_CALLS = 0.5

if GENERATION_MODE == "openai" and FALLBACK_TO_DRY_RUN_WITHOUT_OPENAI_KEY and not os.getenv("OPENAI_API_KEY"):
    GENERATION_MODE = "dry_run"
    print("OPENAI_API_KEY not found; switching GENERATION_MODE to dry_run.")

qa_config = {
    "retrieval_source": RETRIEVAL_SOURCE,
    "retrieval_sources_to_generate": RETRIEVAL_SOURCES_TO_GENERATE,
    "top_k_context": TOP_K_CONTEXT,
    "generation_mode": GENERATION_MODE,
    "generation_model": GENERATION_MODEL,
    "document_known": False
}

qa_config


In [ ]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

QA_DIR = PROCESSED_DIR / "qa_results"
RETRIEVAL_DIR = PROCESSED_DIR / "retrieval_results"

WORKING_DATASET_CSV_PATH = INTERIM_DIR / "financebench_open_source_working.csv"
WORKING_DATASET_PARQUET_PATH = INTERIM_DIR / "financebench_open_source_working.parquet"

QA_DIR.mkdir(parents=True, exist_ok=True)

if RETRIEVAL_SOURCE == "dense":
    RETRIEVAL_RESULTS_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_dense.csv"
    RETRIEVAL_RESULTS_PARQUET_PATH = RETRIEVAL_DIR / "retrieval_results_dense.parquet"
    RUN_NAME = "dense"

elif RETRIEVAL_SOURCE == "hybrid":
    RETRIEVAL_RESULTS_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid.csv"
    RETRIEVAL_RESULTS_PARQUET_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid.parquet"
    RUN_NAME = "hybrid"

elif RETRIEVAL_SOURCE == "hybrid_reranked":
    RETRIEVAL_RESULTS_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid_reranked.csv"
    RETRIEVAL_RESULTS_PARQUET_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid_reranked.parquet"
    RUN_NAME = "hybrid_reranked"

else:
    raise ValueError(f"Unknown RETRIEVAL_SOURCE: {RETRIEVAL_SOURCE}")

QA_RESULTS_CSV_PATH = QA_DIR / f"rag_qa_results_{RUN_NAME}.csv"
QA_RESULTS_PARQUET_PATH = QA_DIR / f"rag_qa_results_{RUN_NAME}.parquet"
QA_MANIFEST_PATH = QA_DIR / f"rag_qa_manifest_{RUN_NAME}.csv"
QA_STATS_PATH = QA_DIR / f"rag_qa_stats_{RUN_NAME}.json"

print("RETRIEVAL_SOURCE:", RETRIEVAL_SOURCE)
print("RUN_NAME        :", RUN_NAME)
print("QA_RESULTS_CSV_PATH:", QA_RESULTS_CSV_PATH)


In [ ]:
if WORKING_DATASET_PARQUET_PATH.exists():
    working_df = pd.read_parquet(WORKING_DATASET_PARQUET_PATH)
elif WORKING_DATASET_CSV_PATH.exists():
    working_df = pd.read_csv(WORKING_DATASET_CSV_PATH)
else:
    raise FileNotFoundError("Working dataset not found.")

print("working_df shape:", working_df.shape)
print(working_df.columns.tolist())
working_df.head(2)


In [ ]:
candidate_paths = [RETRIEVAL_RESULTS_PARQUET_PATH, RETRIEVAL_RESULTS_CSV_PATH]
existing_paths = [p for p in candidate_paths if p.exists()]

if not existing_paths:
    raise FileNotFoundError(f"No retrieval results found for source: {RETRIEVAL_SOURCE}")

RETRIEVAL_RESULTS_PATH = existing_paths[0]

if RETRIEVAL_RESULTS_PATH.suffix == ".parquet":
    retrieval_df = pd.read_parquet(RETRIEVAL_RESULTS_PATH)
else:
    retrieval_df = pd.read_csv(RETRIEVAL_RESULTS_PATH)

print("Using retrieval results:", RETRIEVAL_RESULTS_PATH)
print("retrieval_df shape:", retrieval_df.shape)
print(retrieval_df.columns.tolist())
retrieval_df.head(2)


In [ ]:
required_retrieval_cols = [
    "financebench_id",
    "question",
    "retrieved_rank",
    "chunk_id",
    "retrieved_doc_id",
    "chunk_text",
]

missing_cols = [c for c in required_retrieval_cols if c not in retrieval_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in retrieval_df: {missing_cols}")

print("Retrieval dataframe columns OK.")


In [ ]:
if USE_QUERY_LIMIT:
    keep_ids = working_df["financebench_id"].drop_duplicates().head(QUERY_LIMIT).tolist()
    working_df = working_df[working_df["financebench_id"].isin(keep_ids)].copy().reset_index(drop=True)
    retrieval_df = retrieval_df[retrieval_df["financebench_id"].isin(keep_ids)].copy().reset_index(drop=True)

print("working_df shape after optional limit:", working_df.shape)
print("retrieval_df shape after optional limit:", retrieval_df.shape)


In [ ]:
base_cols = ["financebench_id", "question"]

if "answer" in working_df.columns:
    base_cols.append("answer")

if "doc_name" in working_df.columns:
    base_cols.append("doc_name")

if "company" in working_df.columns:
    base_cols.append("company")

qa_input_df = (
    working_df[base_cols]
    .drop_duplicates(subset=["financebench_id"])
    .copy()
    .reset_index(drop=True)
)

if "answer" not in qa_input_df.columns:
    qa_input_df["answer"] = None

if "doc_name" not in qa_input_df.columns:
    qa_input_df["doc_name"] = None

if "company" not in qa_input_df.columns:
    qa_input_df["company"] = None

print("qa_input_df shape:", qa_input_df.shape)
qa_input_df.head(2)


In [ ]:
def build_context_text(group_df: pd.DataFrame, top_k: int = TOP_K_CONTEXT) -> str:
    top_group = group_df.sort_values("retrieved_rank", ascending=True).head(top_k).copy()

    parts = []
    for _, row in top_group.iterrows():
        retrieved_rank = row.get("retrieved_rank")
        retrieved_doc_id = row.get("retrieved_doc_id")
        chunk_id = row.get("chunk_id")
        chunk_text = str(row.get("chunk_text", "")).strip()

        part = (
            f"[Rank {retrieved_rank} | Doc {retrieved_doc_id} | Chunk {chunk_id}]\n"
            f"{chunk_text}"
        )
        parts.append(part)

    return "\n\n" + ("\n\n" + "-" * 80 + "\n\n").join(parts)


def get_top_chunk_id(group_df: pd.DataFrame):
    top_row = group_df.sort_values("retrieved_rank", ascending=True).iloc[0]
    return top_row.get("chunk_id")


def get_top_doc_id(group_df: pd.DataFrame):
    top_row = group_df.sort_values("retrieved_rank", ascending=True).iloc[0]
    return top_row.get("retrieved_doc_id")


In [ ]:
grouped_retrieval = retrieval_df.groupby("financebench_id", sort=False)

context_records = []

for financebench_id, group in grouped_retrieval:
    context_records.append({
        "financebench_id": financebench_id,
        "context_text": build_context_text(group, top_k=TOP_K_CONTEXT),
        "top_chunk_id": get_top_chunk_id(group),
        "top_doc_id": get_top_doc_id(group),
        "n_retrieved_rows": int(len(group)),
    })

context_df = pd.DataFrame(context_records)

qa_df = qa_input_df.merge(context_df, on="financebench_id", how="left")

print("context_df shape:", context_df.shape)
print("qa_df shape:", qa_df.shape)
qa_df.head(2)


In [ ]:
# ── Τελικό prompt παραγωγής ─────────────────────────────────────────
# Η τελική έκδοση του prompt διαχωρίζει αριθμητικές/πραγματολογικές
# ερωτήσεις από ποιοτικές ερωτήσεις. Έτσι αποφεύγεται η υπερβολική
# συμπίεση απαντήσεων που χρειάζονται σύντομη επεξήγηση.
SYSTEM_PROMPT = """You are a financial analyst answering questions from SEC filings.

Answer the question using ONLY the provided context chunks.

## Output format — choose based on question type

**Numeric or factual questions** (amounts, ratios, dates, yes/no with a single value):
- Lead with the number or value, in the units requested.
- If a calculation is needed, show the formula and intermediate values on one line, then state the final answer.
- Example: 'Operating cash flow ratio = $4,862M / $3,200M = 1.52'

**Qualitative or comparative questions** (trends, comparisons, explanations, outcomes):
- Answer in one or two complete sentences.
- Include the key supporting detail from the context (e.g. the specific percentage, the segment name, the vote outcome phrase).
- Do NOT reduce the answer to a single word or number if the question asks for context or explanation.

## Calculation rules
- You MAY perform arithmetic (addition, subtraction, multiplication, division, percentage change) using numbers explicitly stated in the context.
- You MAY derive ratios, margins, or growth rates if all required line items are present.
- Round to the precision requested in the question (default: 2 decimal places for ratios, 1 for percentages).

## When to refuse
Only say \"Insufficient evidence in the retrieved context.\" when the specific data required is genuinely absent from ALL provided chunks — not merely because the answer requires a calculation.

## Hard rules
- Do not invent numbers or cite sources outside the provided context.
- If a metric is not applicable (e.g. gross margin for a bank), state that and explain briefly using context evidence.
- Do not restate the question in your answer.
"""

def build_user_prompt(question: str, context_text: str) -> str:
    return f"""Context chunks (ranked by relevance):
{context_text}

Question: {question}

Answer:"""


In [ ]:

if GENERATION_MODE == "openai":
    from openai import OpenAI

    openai_api_key = os.getenv("OPENAI_API_KEY")
    if not openai_api_key:
        raise RuntimeError(
            "Δεν βρέθηκε OPENAI_API_KEY στο περιβάλλον εκτέλεσης. "
            "Ορίστε FALLBACK_TO_DRY_RUN_WITHOUT_OPENAI_KEY=True για dry_run."
        )

    openai_client = OpenAI(api_key=openai_api_key)
    print("OpenAI client initialized.")
else:
    openai_client = None
    print("Generation mode: dry_run.")


In [ ]:
def generate_answer(question: str, context_text: str) -> str:
    if GENERATION_MODE == "dry_run":
        return "Η παραγωγή απάντησης παραλείφθηκε σε λειτουργία dry_run."

    # Διόρθωση κωδικοποίησης
    question = question.encode("utf-8", errors="ignore").decode("utf-8")
    context_text = context_text.encode("utf-8", errors="ignore").decode("utf-8")

    user_prompt = build_user_prompt(question=question, context_text=context_text)

    response = openai_client.chat.completions.create(
        model=GENERATION_MODEL,
        temperature=0.0,
        max_completion_tokens=1000,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ]
    )

    answer = response.choices[0].message.content
    if answer:
        return answer.strip()

    return "Insufficient evidence in the retrieved context."


In [ ]:
qa_records = []
manifest_records = []

for idx, row in tqdm(qa_df.iterrows(), total=len(qa_df), desc=f"Running QA for {RUN_NAME}"):
    financebench_id = row["financebench_id"]
    question = row["question"]
    expected_answer = row.get("answer")
    expected_doc_name = row.get("doc_name")
    expected_company = row.get("company")
    context_text = row.get("context_text")
    top_chunk_id = row.get("top_chunk_id")
    top_doc_id = row.get("top_doc_id")
    n_retrieved_rows = row.get("n_retrieved_rows")

    manifest_record = {
        "financebench_id": financebench_id,
        "question": question,
        "expected_doc_name": expected_doc_name,
        "status": None,
        "error_message": None,
        "top_doc_id": top_doc_id,
        "top_chunk_id": top_chunk_id,
    }

    try:
        if pd.isna(context_text) or not str(context_text).strip():
            generated_answer = "Insufficient evidence in the retrieved context."
        else:
            generated_answer = generate_answer(question=question, context_text=context_text)

        qa_records.append({
            "financebench_id": financebench_id,
            "question": question,
            "expected_answer": expected_answer,
            "expected_doc_name": expected_doc_name,
            "expected_company": expected_company,
            "retrieval_source": RETRIEVAL_SOURCE,
            "top_doc_id": top_doc_id,
            "top_chunk_id": top_chunk_id,
            "n_retrieved_rows": n_retrieved_rows,
            "context_text": context_text,
            "generated_answer": generated_answer,
            "generated_answer_len": len(str(generated_answer)),
            "doc_match": str(expected_doc_name) == str(top_doc_id),
        })

        manifest_record["status"] = "success"

    except Exception as e:
        manifest_record["status"] = "error"
        manifest_record["error_message"] = str(e)

    manifest_records.append(manifest_record)

    if GENERATION_MODE == "openai":
        time.sleep(SLEEP_BETWEEN_CALLS)

qa_results_df = pd.DataFrame(qa_records)
qa_manifest_df = pd.DataFrame(manifest_records)

print("qa_results_df shape:", qa_results_df.shape)
print("qa_manifest_df shape:", qa_manifest_df.shape)


In [ ]:
qa_results_df[[
    "financebench_id",
    "question",
    "expected_answer",
    "generated_answer",
    "top_doc_id",
    "expected_doc_name",
    "doc_match"
]].head(10)


In [ ]:
qa_results_df.to_csv(QA_RESULTS_CSV_PATH, index=False, encoding="utf-8")
qa_results_df.to_parquet(QA_RESULTS_PARQUET_PATH, index=False)

qa_manifest_df.to_csv(QA_MANIFEST_PATH, index=False, encoding="utf-8")

print("Αποθηκεύτηκαν:")
print("-", QA_RESULTS_CSV_PATH)
print("-", QA_RESULTS_PARQUET_PATH)
print("-", QA_MANIFEST_PATH)


In [ ]:
successful_generations = int((qa_manifest_df["status"] == "success").sum()) if "status" in qa_manifest_df.columns else 0
failed_generations = int((qa_manifest_df["status"] == "error").sum()) if "status" in qa_manifest_df.columns else 0

qa_stats = {
    "retrieval_source": RETRIEVAL_SOURCE,
    "document_known": False,
    "generation_mode": GENERATION_MODE,
    "generation_model": GENERATION_MODEL,
    "top_k_context": TOP_K_CONTEXT,
    "n_queries": int(qa_df["financebench_id"].nunique()),
    "n_result_rows": int(len(qa_results_df)),
    "n_manifest_rows": int(len(qa_manifest_df)),
    "successful_generations": successful_generations,
    "failed_generations": failed_generations,
    "results_csv": str(QA_RESULTS_CSV_PATH),
    "results_parquet": str(QA_RESULTS_PARQUET_PATH),
    "manifest_csv": str(QA_MANIFEST_PATH),
}

with open(QA_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(qa_stats, f, indent=2, ensure_ascii=False)

print("Saved stats:", QA_STATS_PATH)
qa_stats


In [ ]:
print(qa_manifest_df["status"].value_counts(dropna=False))
qa_manifest_df[["financebench_id", "status", "error_message"]].head(10)



def retrieval_paths_for_source(source: str):
    if source == "dense":
        return (
            RETRIEVAL_DIR / "retrieval_results_dense.parquet",
            RETRIEVAL_DIR / "retrieval_results_dense.csv",
            "dense",
        )
    if source == "hybrid":
        return (
            RETRIEVAL_DIR / "retrieval_results_hybrid.parquet",
            RETRIEVAL_DIR / "retrieval_results_hybrid.csv",
            "hybrid",
        )
    if source == "hybrid_reranked":
        return (
            RETRIEVAL_DIR / "retrieval_results_hybrid_reranked.parquet",
            RETRIEVAL_DIR / "retrieval_results_hybrid_reranked.csv",
            "hybrid_reranked",
        )
    raise ValueError(f"Unknown retrieval source: {source}")


def load_retrieval_results(source: str) -> pd.DataFrame:
    parquet_path, csv_path, _ = retrieval_paths_for_source(source)
    existing = [p for p in [parquet_path, csv_path] if p.exists()]
    if not existing:
        raise FileNotFoundError(f"No retrieval results found for source: {source}")
    path = existing[0]
    df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
    print(f"{source}: loaded {df.shape} from {path}")
    return df


def build_qa_dataframe_for_source(source: str, retrieval_source_df: pd.DataFrame) -> pd.DataFrame:
    missing_cols = [c for c in required_retrieval_cols if c not in retrieval_source_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in {source} retrieval_df: {missing_cols}")

    run_working_df = working_df.copy()
    run_retrieval_df = retrieval_source_df.copy()

    if USE_QUERY_LIMIT:
        keep_ids = run_working_df["financebench_id"].drop_duplicates().head(QUERY_LIMIT).tolist()
        run_working_df = run_working_df[run_working_df["financebench_id"].isin(keep_ids)].copy().reset_index(drop=True)
        run_retrieval_df = run_retrieval_df[run_retrieval_df["financebench_id"].isin(keep_ids)].copy().reset_index(drop=True)

    run_qa_input_df = (
        run_working_df[base_cols]
        .drop_duplicates(subset=["financebench_id"])
        .copy()
        .reset_index(drop=True)
    )

    for optional_col in ["answer", "doc_name", "company"]:
        if optional_col not in run_qa_input_df.columns:
            run_qa_input_df[optional_col] = None

    context_records = []
    for financebench_id, group in run_retrieval_df.groupby("financebench_id", sort=False):
        context_records.append({
            "financebench_id": financebench_id,
            "context_text": build_context_text(group, top_k=TOP_K_CONTEXT),
            "top_chunk_id": get_top_chunk_id(group),
            "top_doc_id": get_top_doc_id(group),
            "n_retrieved_rows": int(len(group)),
        })

    context_source_df = pd.DataFrame(context_records)
    return run_qa_input_df.merge(context_source_df, on="financebench_id", how="left")


def run_qa_generation_for_source(source: str, source_qa_df: pd.DataFrame):
    _, _, run_name = retrieval_paths_for_source(source)

    results_csv = QA_DIR / f"rag_qa_results_{run_name}.csv"
    results_parquet = QA_DIR / f"rag_qa_results_{run_name}.parquet"
    manifest_csv = QA_DIR / f"rag_qa_manifest_{run_name}.csv"
    stats_json = QA_DIR / f"rag_qa_stats_{run_name}.json"

    records = []
    manifests = []

    for _, row in tqdm(source_qa_df.iterrows(), total=len(source_qa_df), desc=f"Running QA for {run_name}"):
        financebench_id = row["financebench_id"]
        question = row["question"]
        expected_answer = row.get("answer")
        expected_doc_name = row.get("doc_name")
        expected_company = row.get("company")
        context_text = row.get("context_text")
        top_chunk_id = row.get("top_chunk_id")
        top_doc_id = row.get("top_doc_id")
        n_retrieved_rows = row.get("n_retrieved_rows")

        manifest_record = {
            "financebench_id": financebench_id,
            "question": question,
            "expected_doc_name": expected_doc_name,
            "status": None,
            "error_message": None,
            "top_doc_id": top_doc_id,
            "top_chunk_id": top_chunk_id,
        }

        try:
            if pd.isna(context_text) or not str(context_text).strip():
                generated_answer = "Insufficient evidence in the retrieved context."
            else:
                generated_answer = generate_answer(question=question, context_text=context_text)

            records.append({
                "financebench_id": financebench_id,
                "question": question,
                "expected_answer": expected_answer,
                "expected_doc_name": expected_doc_name,
                "expected_company": expected_company,
                "retrieval_source": source,
                "top_doc_id": top_doc_id,
                "top_chunk_id": top_chunk_id,
                "n_retrieved_rows": n_retrieved_rows,
                "context_text": context_text,
                "generated_answer": generated_answer,
                "generated_answer_len": len(str(generated_answer)),
                "doc_match": str(expected_doc_name) == str(top_doc_id),
            })
            manifest_record["status"] = "success"
        except Exception as e:
            manifest_record["status"] = "error"
            manifest_record["error_message"] = str(e)

        manifests.append(manifest_record)

        if GENERATION_MODE == "openai":
            time.sleep(SLEEP_BETWEEN_CALLS)

    results_df = pd.DataFrame(records)
    manifest_df = pd.DataFrame(manifests)

    results_df.to_csv(results_csv, index=False, encoding="utf-8")
    results_df.to_parquet(results_parquet, index=False)
    manifest_df.to_csv(manifest_csv, index=False, encoding="utf-8")

    stats = {
        "retrieval_source": source,
        "document_known": False,
        "generation_mode": GENERATION_MODE,
        "generation_model": GENERATION_MODEL,
        "top_k_context": TOP_K_CONTEXT,
        "n_queries": int(source_qa_df["financebench_id"].nunique()),
        "n_result_rows": int(len(results_df)),
        "n_manifest_rows": int(len(manifest_df)),
        "successful_generations": int((manifest_df["status"] == "success").sum()) if "status" in manifest_df.columns else 0,
        "failed_generations": int((manifest_df["status"] == "error").sum()) if "status" in manifest_df.columns else 0,
        "results_csv": str(results_csv),
        "results_parquet": str(results_parquet),
        "manifest_csv": str(manifest_csv),
    }
    with open(stats_json, "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)

    print(f"{run_name}: saved {len(results_df)} QA rows")
    return results_df, manifest_df, stats


generated_run_summaries = {}
for source in RETRIEVAL_SOURCES_TO_GENERATE:
    if source == RETRIEVAL_SOURCE:
        generated_run_summaries[source] = qa_stats
        continue

    source_retrieval_df = load_retrieval_results(source)
    source_qa_df = build_qa_dataframe_for_source(source, source_retrieval_df)
    _, _, source_stats = run_qa_generation_for_source(source, source_qa_df)
    generated_run_summaries[source] = source_stats

generated_run_summaries
